# Week 4: Risk-Neutral Valuation of a Stylized GMAB under GBM

**Project:** *Valuing Variable Annuity Guarantees Under Market Crashes and Regime Shifts*  
**Mentor:** Hao Quan  
**Notebook type:** all-in-one lesson, lab, exercises, and worked solutions

> **Driving question:** What is the present value of the insurer's maturity top-up in a smooth market?

## Learning goals

- Express a return-of-premium GMAB as a put-like payoff.
- Explain why valuation uses discounted expectation under a pricing measure.
- Estimate value, standard error, and a confidence interval by Monte Carlo.
- Validate Monte Carlo against the Black–Scholes put formula.
- Measure sensitivity to volatility, fees, maturity, and guarantee level.

## How to use this notebook

1. Complete the assigned reading before the weekly meeting.
2. Read the explanation cells and predict each result before running the code.
3. Run the notebook from top to bottom. Every random experiment uses a fixed seed.
4. Attempt the exercises before opening the worked-solution section.
5. Write a 150–250 word interpretation of the main result in your own words.

This notebook is educational. It simplifies real contracts and is not an insurance
quotation, investment recommendation, or complete actuarial valuation.

## Assigned reading

- [NYU Continuous-Time Finance Notes, Section 3](https://math.nyu.edu/faculty/kohn/cont-time-finance/section3.pdf) — read pp. 1–2 for the risk-neutral measure and discounted pricing identity
- [Penn Freedom GMAB section](https://www.sec.gov/Archives/edgar/data/702184/000119312522115288/d269677d485bpos.htm) — revisit pp. 51–53 and identify the maturity top-up


In [ ]:
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
np.set_printoptions(precision=4, suppress=True)

SEED = 20260730
rng = np.random.default_rng(SEED)
print(f"Reproducible random seed: {SEED}")


## 1. Common stylized contract

| Parameter | Symbol | Baseline |
|---|---:|---:|
| Initial premium/account | $A_0=P$ | $100,000 |
| Maturity | $T$ | 10 years |
| Risk-free rate | $r$ | 3% |
| Total continuous account charge | $c$ | 2% |
| Volatility | $\sigma$ | 16% |
| GMAB amount | $G$ | $100,000 |

We omit mortality, lapse, surrender, withdrawals, additional premiums,
reset/step-up rules, taxes, hedging costs, capital, and profit.

Under the selected pricing measure $Q$,

$$dA_t=(r-c)A_tdt+\sigma A_tdW_t^Q.$$

The gross guarantee value is

$$V_0=e^{-rT}E^Q[(G-A_T)^+].$$

This is the incremental guarantee—not the value of the full VA.


In [ ]:
@dataclass(frozen=True)
class GMABContract:
    A0: float = 100_000.0
    G: float = 100_000.0
    T: float = 10.0
    r: float = 0.03
    charge: float = 0.02
    sigma: float = 0.16


contract = GMABContract()
print(contract)


## 2. Closed-form benchmark


In [ ]:
def gmab_gbm_closed_form(A0, G, T, r, charge, sigma):
    if A0 <= 0 or G < 0 or T < 0 or sigma < 0:
        raise ValueError("invalid contract inputs")
    if G == 0:
        return 0.0
    if T == 0:
        return max(G - A0, 0.0)
    if sigma == 0:
        deterministic_account = A0 * np.exp((r - charge) * T)
        return np.exp(-r * T) * max(G - deterministic_account, 0.0)
    root_T = np.sqrt(T)
    d1 = (np.log(A0 / G) + (r - charge + 0.5 * sigma**2) * T) / (sigma * root_T)
    d2 = d1 - sigma * root_T
    return G * np.exp(-r * T) * norm.cdf(-d2) - A0 * np.exp(-charge * T) * norm.cdf(-d1)


analytic_value = gmab_gbm_closed_form(**contract.__dict__)
print(f"Closed-form GMAB value: ${analytic_value:,.2f}")
assert 0 <= analytic_value <= contract.G * np.exp(-contract.r * contract.T)


## 3. Monte Carlo estimator

For independent terminal simulations $A_T^{(i)}$, define discounted losses

$$X_i=e^{-rT}(G-A_T^{(i)})^+.$$

Then $\widehat V_0=\bar X$, with

$$\widehat{SE}=s_X/\sqrt{M}.$$


In [ ]:
def simulate_gbm_terminal_q(A0, T, r, charge, sigma, n_paths, seed=SEED, z=None):
    if z is None:
        generator = np.random.default_rng(seed)
        z = generator.standard_normal(n_paths)
    z = np.asarray(z)
    if z.shape != (n_paths,):
        raise ValueError("z must have shape (n_paths,)")
    return A0 * np.exp(
        (r - charge - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * z
    )


def summarize_gmab(terminal_accounts, G, T, r):
    terminal_accounts = np.asarray(terminal_accounts)
    payoff = np.maximum(G - terminal_accounts, 0.0)
    discounted = np.exp(-r * T) * payoff
    n = len(discounted)
    price = discounted.mean()
    se = discounted.std(ddof=1) / np.sqrt(n)
    trigger = payoff > 0
    conditional_shortfall = payoff[trigger].mean() if trigger.any() else 0.0
    return {
        "n_paths": n,
        "price": price,
        "se": se,
        "ci_low": price - 1.96 * se,
        "ci_high": price + 1.96 * se,
        "trigger_probability": trigger.mean(),
        "conditional_undiscounted_shortfall": conditional_shortfall,
    }, discounted


n_paths = 200_000
terminal = simulate_gbm_terminal_q(
    contract.A0, contract.T, contract.r, contract.charge,
    contract.sigma, n_paths, seed=SEED + 1
)
summary, discounted_losses = summarize_gmab(
    terminal, contract.G, contract.T, contract.r
)
print(pd.Series(summary).to_string())
print(f"Analytic value: ${analytic_value:,.2f}")
print(f"MC minus analytic: ${summary['price'] - analytic_value:,.2f}")

assert abs(summary["price"] - analytic_value) < 4 * summary["se"]


## 4. See the mass at zero and the loss tail


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(terminal, bins=90, alpha=0.75)
axes[0].axvline(contract.G, color="black", linestyle="--", label="Guarantee")
axes[0].set(title="Terminal account under $Q$", xlabel="Terminal dollars", ylabel="Frequency")
axes[0].legend()

positive_losses = discounted_losses[discounted_losses > 0]
axes[1].hist(positive_losses, bins=70, alpha=0.75, color="#B85C38")
axes[1].set(
    title="Discounted GMAB loss conditional on a claim",
    xlabel="Present-value dollars",
    ylabel="Frequency",
)
plt.tight_layout()
plt.show()


## 5. Convergence and the $M^{-1/2}$ rule


In [ ]:
conv_rng = np.random.default_rng(SEED + 2)
z_bank = conv_rng.standard_normal(200_000)
conv_rows = []
for n in [1_000, 5_000, 20_000, 80_000, 200_000]:
    A_T = simulate_gbm_terminal_q(
        contract.A0, contract.T, contract.r, contract.charge,
        contract.sigma, n, z=z_bank[:n]
    )
    stats, _ = summarize_gmab(A_T, contract.G, contract.T, contract.r)
    conv_rows.append([
        n, stats["price"], stats["se"], stats["ci_low"], stats["ci_high"],
        abs(stats["price"] - analytic_value)
    ])

convergence = pd.DataFrame(
    conv_rows,
    columns=["n", "estimate", "se", "ci_low", "ci_high", "absolute_error"],
)
print(convergence.to_string(index=False))

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.errorbar(
    convergence["n"], convergence["estimate"],
    yerr=1.96 * convergence["se"], fmt="o-", capsize=3, label="Monte Carlo 95% CI"
)
ax.axhline(analytic_value, color="black", linestyle="--", label="Closed form")
ax.set_xscale("log")
ax.set(title="Monte Carlo convergence", xlabel="Number of paths (log scale)", ylabel="GMAB value")
ax.legend()
plt.show()


## 6. Sensitivity analysis with an exact benchmark

For this stylized payoff, guarantee value should generally increase with
volatility and guarantee generosity. A higher account charge lowers the
policyholder's account and therefore increases the **gross** top-up value.
That does not by itself determine whether a rider charge is profitable because
fee revenue is outside this gross-liability calculation.


In [ ]:
vol_grid = np.linspace(0.05, 0.35, 13)
fee_grid = np.linspace(0.00, 0.035, 8)
vol_values = [
    gmab_gbm_closed_form(contract.A0, contract.G, contract.T, contract.r, contract.charge, s)
    for s in vol_grid
]
fee_values = [
    gmab_gbm_closed_form(contract.A0, contract.G, contract.T, contract.r, c, contract.sigma)
    for c in fee_grid
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(vol_grid, vol_values, marker="o")
axes[0].set(title="Sensitivity to volatility", xlabel="Volatility", ylabel="GMAB value")
axes[1].plot(fee_grid, fee_values, marker="o", color="#8B5A9F")
axes[1].set(title="Gross liability sensitivity to account charge", xlabel="Continuous charge", ylabel="GMAB value")
plt.tight_layout()
plt.show()

assert np.all(np.diff(vol_values) > 0)
assert np.all(np.diff(fee_values) > 0)


In [ ]:
maturity_grid = np.array([5, 7, 10, 15, 20])
guarantee_ratios = np.array([0.80, 0.90, 1.00, 1.10, 1.20])
sensitivity_matrix = np.empty((len(maturity_grid), len(guarantee_ratios)))
for i, maturity in enumerate(maturity_grid):
    for j, ratio in enumerate(guarantee_ratios):
        sensitivity_matrix[i, j] = gmab_gbm_closed_form(
            contract.A0, contract.A0 * ratio, maturity,
            contract.r, contract.charge, contract.sigma
        )

sens_df = pd.DataFrame(
    sensitivity_matrix,
    index=[f"T={t}" for t in maturity_grid],
    columns=[f"G/A0={x:.1f}" for x in guarantee_ratios],
)
print(sens_df.round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(sensitivity_matrix, aspect="auto", cmap="viridis")
ax.set_xticks(np.arange(len(guarantee_ratios)), [f"{x:.1f}" for x in guarantee_ratios])
ax.set_yticks(np.arange(len(maturity_grid)), maturity_grid)
ax.set(xlabel="Guarantee / initial account", ylabel="Maturity (years)", title="GMAB value sensitivity")
fig.colorbar(im, ax=ax, label="Present-value dollars")
plt.show()


## 7. Exercises

1. Revalue the guarantee at 25% volatility and report the dollar and percentage
   increase from baseline.
2. Use antithetic normals $(Z,-Z)$ and compare the standard error with ordinary
   Monte Carlo using the same number of terminal values.
3. Explain why the insurer liability is $(G-A_T)^+$ rather than
   $\max(G,A_T)$.
4. Identify at least five omitted real-contract features.


## 8. Worked solutions


In [ ]:
# Exercise 1
value_25 = gmab_gbm_closed_form(
    contract.A0, contract.G, contract.T, contract.r, contract.charge, 0.25
)
print(f"Value at 25% volatility: ${value_25:,.2f}")
print(f"Increase from baseline: ${value_25 - analytic_value:,.2f} ({value_25 / analytic_value - 1:.1%})")

# Exercise 2
anti_rng = np.random.default_rng(SEED + 3)
half = 50_000
z_half = anti_rng.standard_normal(half)
z_antithetic = np.concatenate([z_half, -z_half])
A_anti = simulate_gbm_terminal_q(
    contract.A0, contract.T, contract.r, contract.charge,
    contract.sigma, 2 * half, z=z_antithetic
)
ordinary_rng = np.random.default_rng(SEED + 4)
A_ordinary = simulate_gbm_terminal_q(
    contract.A0, contract.T, contract.r, contract.charge,
    contract.sigma, 2 * half, z=ordinary_rng.standard_normal(2 * half)
)
anti_stats, _ = summarize_gmab(A_anti, contract.G, contract.T, contract.r)
ordinary_stats, _ = summarize_gmab(A_ordinary, contract.G, contract.T, contract.r)
print(pd.DataFrame([
    ["ordinary", ordinary_stats["price"], ordinary_stats["se"]],
    ["antithetic terminal sample", anti_stats["price"], anti_stats["se"]],
], columns=["method", "price", "naive_individual_SE"]).to_string(index=False))
print("Note: a proper paired antithetic SE treats each (Z,-Z) average as one observation.")

paired_losses = np.exp(-contract.r * contract.T) * 0.5 * (
    np.maximum(contract.G - A_anti[:half], 0)
    + np.maximum(contract.G - A_anti[half:], 0)
)
paired_se = paired_losses.std(ddof=1) / np.sqrt(half)
print(f"Proper paired antithetic SE: {paired_se:.4f}")
assert abs(anti_stats["price"] - analytic_value) < 4 * paired_se


## 9. Model and interpretation limits

- A risk-neutral price is not a forecast of the average future claim.
- The gross guarantee value is not a quoted rider fee.
- The complete contract has fees, behavior, mortality, investment restrictions,
  and termination rules that are absent here.
- In GBM, all paths are continuous. That assumption is challenged next week.

### Weekly submission

- Completed notebook
- One-page sensitivity note
- Validation table comparing Monte Carlo and closed form
